# # 🚀 Triển Khai Suy Luận VidVRD Với Mô Hình Qwen2.5-VL-3B-Instruct (Alibaba Video-LLM)
# 
# **Đề tài**: Video Visual Relation Detection (VidVRD) trên Video Giám sát Thực tế
# **Mô hình VLM**: `Qwen/Qwen2.5-VL-3B-Instruct` (Chuyên dụng cho Video, 3B tham số, suy luận 15s trên GPU T4)
# **Kỹ thuật**: Set-of-Marks (SoM) + Căn chỉnh 60 S/Objects & 26 Relations + Giải thích Reason


In [ ]:
# ============================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & KIỂM TRA GPU T4 (TƯƠNG THÍCH QWEN 2.5-VL)
# ============================================================
# 1. Gỡ bỏ gói torchaudio thừa để triệt tiêu hoàn toàn xung đột phiên bản CUDA
!pip uninstall -y torchaudio -q

# 2. Cài đặt các thư viện cần thiết cho Qwen 2.5-VL Vision-Language Model
!pip install -q --upgrade transformers accelerate qwen-vl-utils

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu: Thời lượng chạy (Runtime) -> Thay đổi loại thời lượng chạy -> Chọn T4 GPU.")


In [ ]:
# ============================================================
# BƯỚC 2: NẠP MÔ HÌNH QWEN 2.5-VL 3B INSTRUCT (CHUYÊN DỤNG VIDEO & THỊ GIÁC)
# ============================================================
import sys
import torch

# Phòng thủ: Vô hiệu hóa torchaudio nếu phát hiện lỗi xung đột
try:
    import torchaudio
except Exception:
    try:
        import transformers.utils.import_utils as _iu
        _iu._torchaudio_available = False
    except Exception:
        pass

try:
    from transformers import Qwen2_5_VLForConditionalGeneration as ModelClass
except ImportError:
    from transformers import AutoModelForVision2Seq as ModelClass

from transformers import AutoProcessor

# Sử dụng chính thức mô hình Qwen2.5-VL-3B-Instruct chuyên biệt cho Video-LLM từ Alibaba
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
print(f"⏳ Đang nạp mô hình {model_id} (Trọng số ~6 GB)...")

model = ModelClass.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    model_id
)

print(f"✅ Mô hình {model_id} đã nạp thành công vào GPU T4 với kiến trúc Video-LLM!")


In [ ]:
# ============================================================
# BƯỚC 3: SUY LUẬN VLM TỰ ĐỘNG (CHUẨN HỌC THUẬT VIDVRD + GIẢI THÍCH REASON)
# ============================================================
import os
import json
import glob
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info

# 1. Tự động tìm 8 frames ảnh visual prompt trong /content/
all_imgs = sorted(glob.glob("/content/*.jpg") + glob.glob("/content/frames/*.jpg"))
image_paths = [p for p in all_imgs if "vlm_" in os.path.basename(p) or "frame" in os.path.basename(p)]
if not image_paths:
    image_paths = all_imgs[:8]
if not image_paths:
    raise FileNotFoundError("⚠️ Không tìm thấy ảnh .jpg nào trong /content/. Hãy tải 8 ảnh frame lên!")

print(f"✅ Đã tìm thấy {len(image_paths)} frames ảnh visual prompt.")

# 2. Tự động đọc Prompt & Từ vựng từ file Payload JSON
payload_files = glob.glob("/content/*payload*.json") + glob.glob("/content/*.json")
payload_files = [p for p in payload_files if "ket_qua" not in p]

if payload_files:
    print(f"📄 Tự động tải cấu hình từ payload: {payload_files[0]}")
    with open(payload_files[0], "r", encoding="utf-8") as f:
        payload_data = json.load(f)
    system_prompt = payload_data.get("vlm_system_prompt", "")
    user_prompt = payload_data.get("vlm_user_prompt", "")
    allowed_relations = payload_data.get("allowed_relations_vocabulary_26", [])
else:
    print("ℹ️ Dùng prompt mặc định tổng quát (chuẩn 26 relations + reason):")
    allowed_relations = ['bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit', 'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)', 'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave']
    system_prompt = (
        "You are an advanced Video Visual Relation Detection (VidVRD) AI for surveillance analytics. "
        "You are given a temporal sequence of video frames with numbered visual marks [ID] identifying subjects and objects. "
        "Your task is to detect all active visual relations occurring between the marked entities over time.\n\n"
        "STRICT CONSTRAINTS:\n"
        f"1. You MUST strictly select relation predicates ONLY from these 26 predefined categories: {allowed_relations}.\n"
        "2. SYSTEMATIC INTERACTION RULES:\n"
        "   - Person-Person interactions: Look for active physical contact or greeting (such as touch, hug, wave).\n"
        "   - Person-Object interactions: Only predict hold or carry if a person is actively holding or carrying the object in their hands. If an object is resting on the floor and a person merely walks past it without touching, DO NOT predict any relation.\n"
        "   - Vehicle predicates (get_on, get_off, ride, drive) apply ONLY to vehicles or riding animals, NEVER to handheld items like handbags or backpacks.\n"
        "3. Output format MUST be strictly a valid JSON object matching this schema:\n"
        "{\n"
        '  "temporal_summary": "<brief 1-sentence description of overall interactions and movements across frames>",\n'
        '  "triplets": [\n'
        '    {\n'
        '      "subject": "[ID]",\n'
        '      "relation": "<predicate>",\n'
        '      "object": "[ID]",\n'
        '      "reason": "<brief explanation of why this relation is selected based on visual evidence>"\n'
        '    }\n'
        '  ]\n'
        "}\n"
        "4. DO NOT output any markdown code blocks, explanations, or conversational text. Output ONLY raw JSON."
    )
    user_prompt = (
        "Analyze the provided sequential frames of this surveillance video clip.\n"
        "Perform a systematic pair-by-pair check across the full time duration:\n"
        "- Examine Person-Person interactions: check if [1] and [2] touch, hug, or wave.\n"
        "- Examine Person-Object interactions: check if any person is actively holding or carrying [4].\n"
        "First write a brief 1-sentence temporal_summary of observed actions, then list all detected relation triplets with a clear reason for each.\n"
        "Select predicates strictly from the allowed 26 categories. "
        'Respond strictly with the JSON object: {"temporal_summary": "...", "triplets": [{"subject": "[ID]", "relation": "<verb>", "object": "[ID]", "reason": "..."}]}.'
    )

# 3. Chuẩn bị nội dung tuần tự kèm mốc thời gian rõ ràng (Interleaved Temporal Anchoring)
user_content = []
for idx, p in enumerate(image_paths, 1):
    base_fn = os.path.basename(p)
    ts = base_fn.split("_")[-1].replace(".jpg", "") if "_" in base_fn and "s.jpg" in base_fn else f"{idx}s"
    user_content.append({"type": "text", "text": f"[Frame {idx} at timestamp {ts}]:"})
    user_content.append({"type": "image", "image": p})
user_content.append({"type": "text", "text": "\n" + user_prompt})

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# 4. Xử lý dữ liệu đầu vào và suy luận trên GPU T4 (Siêu tốc 15s)
text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

proc_kwargs = {"text": [text_prompt], "padding": True, "return_tensors": "pt"}
if image_inputs is not None and len(image_inputs) > 0:
    proc_kwargs["images"] = image_inputs
if video_inputs is not None and len(video_inputs) > 0:
    proc_kwargs["videos"] = video_inputs

inputs = processor(**proc_kwargs).to("cuda")

with torch.no_grad():
    # Pure Greedy Search: Bảo toàn token khóa JSON chính xác
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=False
    )

generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
raw_output = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

# 5. Trích xuất và kiểm tra mảng JSON kết quả kèm cột Reason
clean_text = raw_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
try:
    parsed_json = json.loads(clean_text)

    if isinstance(parsed_json, dict):
        summary = parsed_json.get("temporal_summary", "")
        if summary:
            print(f"🎬 Quan sát thời gian (Temporal Summary): {summary}\n")
        triplets = parsed_json.get("triplets", [])
    elif isinstance(parsed_json, list):
        triplets = parsed_json
    else:
        triplets = []

    # Lưu kết quả độc lập ra file JSON
    with open("/content/ket_qua_vlm.json", "w", encoding="utf-8") as f:
        json.dump(triplets, f, indent=2, ensure_ascii=False)

    print("=" * 80)
    print("KẾT QUẢ SUY LUẬN VLM (PREDICTED RELATION TRIPLETS + REASONING):")
    print("=" * 80)
    print(json.dumps(triplets, indent=2, ensure_ascii=False))
    print("-" * 80)
    print(f"{'Subject':<10} | {'Relation':<16} | {'Object':<10} | {'Status':<10} | Reason / Explanation")
    print("-" * 80)
    for t in triplets:
        sub = t.get("subject", "")
        rel = t.get("relation", "")
        obj = t.get("object", "")
        reason = t.get("reason", "N/A")
        is_valid = rel in allowed_relations if allowed_relations else True
        status = "[OK]" if is_valid else "[X] Invalid"
        print(f"{sub:<10} | {rel:<16} | {obj:<10} | {status:<10} | {reason}")
    print("=" * 80)
    print(f"✅ Đã lưu kết quả tự động vào: /content/ket_qua_vlm.json ({len(triplets)} triplets)")
except Exception as e:
    print("Raw output từ mô hình:", raw_output)
    print(f"Lỗi phân tích JSON: {e}")
